In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
!pip install transformers==4.46.3
!pip install accelerate sentencepiece psutil

In [ ]:
import torch
import time
import psutil

from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModel
)

print("=" * 60)
print("HUGGING FACE AI WORKSTATION")
print("=" * 60)

# ============================================================
# DEVICE INFORMATION
# ============================================================

device = 0 if torch.cuda.is_available() else -1

print("\nDEVICE INFORMATION")

if torch.cuda.is_available():
    print("Device: CUDA")
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Device: CPU")

# ============================================================
# TOKENIZATION
# ============================================================

print("\nTOKENIZATION")

gpt2_tokenizer = AutoTokenizer.from_pretrained(
    "distilgpt2"
)

text = """
Artificial Intelligence is transforming
education and cybersecurity.
"""

tokens = gpt2_tokenizer.tokenize(text)

print("Tokens:")
print(tokens)

print("Token Count:", len(tokens))

# ============================================================
# EMBEDDINGS
# ============================================================

print("\nEMBEDDINGS")

bert_tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

bert_model = AutoModel.from_pretrained(
    "bert-base-uncased"
)

inputs = bert_tokenizer(
    text,
    return_tensors="pt",
    padding=True,
    truncation=True
)

with torch.no_grad():
    outputs = bert_model(**inputs)

embedding = outputs.last_hidden_state

print("Embedding Shape:", embedding.shape)

# ============================================================
# TEXT GENERATION
# ============================================================

print("\nTEXT GENERATION")

generator = pipeline(
    "text-generation",
    model="distilgpt2",
    device=device
)

result = generator(
    "Machine Learning is",
    max_new_tokens=50,
    pad_token_id=50256
)

print(result[0]["generated_text"])

# ============================================================
# SENTIMENT ANALYSIS
# ============================================================

print("\nSENTIMENT ANALYSIS")

sentiment = pipeline(
    "sentiment-analysis",
    device=device
)

reviews = [
    "This workshop is amazing.",
    "The session was boring."
]

for review in reviews:
    print("\nReview:", review)
    print(sentiment(review))

# ============================================================
# SUMMARIZATION
# ============================================================

print("\nSUMMARIZATION")

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=device
)

article = """
Artificial Intelligence is a branch of computer science
focused on creating systems capable of performing tasks
that typically require human intelligence.

AI enables learning, reasoning, decision-making,
automation, prediction, and language understanding.

It is transforming healthcare, education, finance,
cybersecurity, and many other industries.
"""

summary = summarizer(
    article,
    max_length=40,
    min_length=10,
    do_sample=False
)

print(summary[0]["summary_text"])

# ============================================================
# BENCHMARK
# ============================================================

print("\nBENCHMARK")

prompt = "Explain Transformers in AI."

start = time.time()

output = generator(
    prompt,
    max_new_tokens=100,
    pad_token_id=50256
)

end = time.time()

generated_text = output[0]["generated_text"]

tokens_generated = len(
    generated_text.split()
)

throughput = (
    tokens_generated /
    (end - start)
)

print(
    "Execution Time:",
    round(end - start, 2),
    "seconds"
)

print(
    "Tokens Per Second:",
    round(throughput, 2)
)

# ============================================================
# MEMORY MONITORING
# ============================================================

print("\nMEMORY")

process = psutil.Process()

memory = (
    process.memory_info().rss /
    1024**2
)

print(
    "RAM Usage:",
    round(memory, 2),
    "MB"
)

# ============================================================
# GPU MONITORING
# ============================================================

if torch.cuda.is_available():

    gpu_memory = (
        torch.cuda.memory_allocated()
        / 1024**3
    )

    print(
        "GPU Memory:",
        round(gpu_memory, 2),
        "GB"
    )

print("\nExecution Complete")
print("=" * 60)